# LLM From Scratch — Full Training & Testing on Google ColabThis notebook runs the **complete pipeline**:1. **Model Architecture** — GPT-2 style transformer built from scratch2. **Wikipedia Scraper** — Fetches technology articles3. **Training** — Pre-trains a small GPT on scraped data (GPU)4. **Fine-Tuning** — Fine-tunes pre-trained GPT-2 124M on tech data5. **Interactive Testing** — Chat with your trained model> **Runtime**: Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# Check GPU availabilityimport torchprint(f"PyTorch version: {torch.__version__}")print(f"CUDA available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")    device = torch.device("cuda")else:    print("WARNING: No GPU found. Training will be slow.")    device = torch.device("cpu")print(f"Using device: {device}")

In [ ]:
# Install dependencies!pip install tiktoken -q

## Part 1: Model Architecture (Built From Scratch)All components below are the same as in your local modular code.

In [ ]:
import torchimport torch.nn as nnimport tiktokenimport timeimport os# ========== GELU Activation ==========class GELU(nn.Module):    def __init__(self):        super().__init__()    def forward(self, x):        return 0.5 * x * (1 + torch.tanh(            torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))        ))# ========== Layer Normalization ==========class LayerNorm(nn.Module):    def __init__(self, emb_dim):        super().__init__()        self.eps = 1e-5        self.scale = nn.Parameter(torch.ones(emb_dim))        self.shift = nn.Parameter(torch.zeros(emb_dim))    def forward(self, x):        mean = x.mean(dim=-1, keepdim=True)        var = x.var(dim=-1, keepdim=True, unbiased=False)        norm_x = (x - mean) / torch.sqrt(var + self.eps)        return self.scale * norm_x + self.shift# ========== Feed-Forward Network ==========class FeedForward(nn.Module):    def __init__(self, cfg):        super().__init__()        self.layer = nn.Sequential(            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),            GELU(),            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),        )    def forward(self, x):        return self.layer(x)# ========== Multi-Head Attention ==========class MultiHeadAttention(nn.Module):    def __init__(self, d_in, d_out, context_length, dropout, num_head, qkv_bias=False):        super().__init__()        assert d_out % num_head == 0, "d_out must be divisible by num_head"        self.d_out = d_out        self.num_head = num_head        self.head_dim = d_out // num_head        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)        self.w_key = nn.Linear(d_in, d_out, bias=qkv_bias)        self.w_value = nn.Linear(d_in, d_out, bias=qkv_bias)        self.out_proj = nn.Linear(d_out, d_out)        self.dropout = nn.Dropout(dropout)        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))    def forward(self, x):        b, num_token, d_in = x.shape        key = self.w_key(x)        query = self.w_query(x)        value = self.w_value(x)        key = key.view(b, num_token, self.num_head, self.head_dim).transpose(1, 2)        query = query.view(b, num_token, self.num_head, self.head_dim).transpose(1, 2)        value = value.view(b, num_token, self.num_head, self.head_dim).transpose(1, 2)        attention_score = query @ key.transpose(2, 3)        mask_bool = self.mask.bool()[:num_token, :num_token]        attention_score = attention_score.masked_fill(mask_bool, -torch.inf)        attention_weight = torch.softmax(attention_score / key.shape[-1] ** 0.5, dim=-1)        attention_weight = self.dropout(attention_weight)        context_vec = (attention_weight @ value).transpose(1, 2)        context_vec = context_vec.contiguous().view(b, num_token, self.d_out)        context_vec = self.out_proj(context_vec)        return context_vec# ========== Transformer Block ==========class TransformerBlock(nn.Module):    def __init__(self, cfg):        super().__init__()        self.attention = MultiHeadAttention(            d_in=cfg["emb_dim"], d_out=cfg["emb_dim"],            context_length=cfg["context_length"], num_head=cfg["n_heads"],            dropout=cfg["drop_rate"], qkv_bias=cfg["qkv_bias"]        )        self.ff = FeedForward(cfg)        self.norm1 = LayerNorm(cfg["emb_dim"])        self.norm2 = LayerNorm(cfg["emb_dim"])        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])    def forward(self, x):        shortcut = x        x = self.norm1(x)        x = self.attention(x)        x = self.drop_shortcut(x)        x = x + shortcut        shortcut = x        x = self.norm2(x)        x = self.ff(x)        x = self.drop_shortcut(x)        x = x + shortcut        return x# ========== GPT Model ==========class GPTModel(nn.Module):    def __init__(self, cfg):        super().__init__()        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])        self.drop_emb = nn.Dropout(cfg["drop_rate"])        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])        self.final_norm = LayerNorm(cfg["emb_dim"])        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)    def forward(self, in_index):        batch_size, seq_len = in_index.shape        tok_embeds = self.tok_emb(in_index)        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_index.device))        x = tok_embeds + pos_embeds        x = self.drop_emb(x)        x = self.trf_blocks(x)        x = self.final_norm(x)        logits = self.out_head(x)        return logitsprint("Model architecture loaded successfully!")

## Part 2: Training Utilities

In [ ]:
# ========== Dataset & DataLoader ==========from torch.utils.data import Dataset, DataLoaderclass GPTDatasetV1(Dataset):    def __init__(self, txt, tokenizer, max_len, stride):        self.input_ids = []        self.target_ids = []        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})        for i in range(0, len(token_ids) - max_len, stride):            input_chunk = token_ids[i : i + max_len]            target_chunk = token_ids[i + 1 : i + max_len + 1]            self.input_ids.append(torch.tensor(input_chunk))            self.target_ids.append(torch.tensor(target_chunk))    def __len__(self):        return len(self.input_ids)    def __getitem__(self, idx):        return self.input_ids[idx], self.target_ids[idx]def create_dataloader(txt, tokenizer, batch_size=4, max_len=256, stride=128,                      shuffle=True, drop_last=True):    dataset = GPTDatasetV1(txt, tokenizer, max_len, stride)    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,                      drop_last=drop_last, num_workers=0)# ========== Text Helpers ==========def text_to_token(text, tokenizer):    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})    return torch.tensor(encoded, dtype=torch.long).unsqueeze(0)def token_to_text(token_ids, tokenizer):    return tokenizer.decode(token_ids.squeeze(0).tolist())# ========== Generation ==========def generate_text(model, idx, max_new_token, context_size, temperature=0.7, top_k=50):    model.eval()    for _ in range(max_new_token):        idx_cond = idx[:, -context_size:]        with torch.no_grad():            logits = model(idx_cond)        logits = logits[:, -1, :]        if temperature > 0:            logits = logits / temperature            if top_k > 0:                top_vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))                min_val = top_vals[:, -1].unsqueeze(-1)                logits = torch.where(logits < min_val, torch.full_like(logits, float('-inf')), logits)            probs = torch.softmax(logits, dim=-1)            idx_next = torch.multinomial(probs, num_samples=1)        else:            idx_next = torch.argmax(logits, dim=-1, keepdim=True)        if idx_next.item() == 50256:  # EOT token            break        idx = torch.cat((idx, idx_next), dim=1)    return idx# ========== Training Functions ==========def cal_loss_batch(input_batch, target_batch, model, device):    input_batch, target_batch = input_batch.to(device), target_batch.to(device)    logits = model(input_batch)    return torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())def cal_loss_loader(data_loader, model, device, num_batches=None):    total_loss = 0.0    if len(data_loader) == 0:        return float("nan")    if num_batches is None:        num_batches = len(data_loader)    else:        num_batches = min(num_batches, len(data_loader))    for i, (inp, tgt) in enumerate(data_loader):        if i < num_batches:            total_loss += cal_loss_batch(inp, tgt, model, device).item()        else:            break    return total_loss / num_batchesdef evaluate_model(model, train_loader, val_loader, device, eval_iter):    model.eval()    with torch.no_grad():        train_loss = cal_loss_loader(train_loader, model, device, num_batches=eval_iter)        val_loss = cal_loss_loader(val_loader, model, device, num_batches=eval_iter)    model.train()    return train_loss, val_lossdef train_model(model, train_loader, val_loader, optimizer, device, num_epoch,                eval_freq, eval_iter, start_context, tokenizer):    train_losses, val_losses = [], []    tokens_seen, global_step = 0, -1    for epoch in range(num_epoch):        model.train()        for input_batch, target_batch in train_loader:            optimizer.zero_grad()            loss = cal_loss_batch(input_batch, target_batch, model, device)            loss.backward()            optimizer.step()            tokens_seen += input_batch.numel()            global_step += 1            if global_step % eval_freq == 0:                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)                train_losses.append(train_loss)                val_losses.append(val_loss)                print(f"Ep {epoch+1} (step{global_step:06d}): Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")        # Generate sample after each epoch        model.eval()        ctx = text_to_token(start_context, tokenizer).to(device)        with torch.no_grad():            out = generate_text(model, ctx, max_new_token=50, context_size=model.pos_emb.weight.shape[0])        print(f"[Epoch {epoch+1}] {token_to_text(out, tokenizer)}")    return train_losses, val_lossesprint("Training utilities loaded!")

## Part 3: Wikipedia Scraper for Technology Data

In [ ]:
import urllib.requestimport urllib.parseimport jsonimport reTECHNOLOGY_TOPICS = [    "Artificial intelligence", "Machine learning", "Deep learning",    "Neural network", "Natural language processing", "Computer science",    "Computer programming", "Software engineering", "Internet",    "Cloud computing", "Blockchain", "Cybersecurity", "Robotics",    "Quantum computing", "Internet of things", "Big data", "Data science",    "Algorithm", "Operating system", "Python (programming language)",    "Semiconductor", "Microprocessor", "Graphics processing unit",    "Virtual reality", "Computer vision", "Reinforcement learning",    "Convolutional neural network", "Recurrent neural network",    "Transformer (deep learning architecture)", "Database",]def fetch_wikipedia_article(title):    encoded_title = urllib.parse.quote(title)    url = (f"https://en.wikipedia.org/w/api.php?"           f"action=query&titles={encoded_title}"           f"&prop=extracts&explaintext=1&format=json")    try:        req = urllib.request.Request(url, headers={            "User-Agent": "LLM-Scratch-Colab/1.0 (Educational)"        })        with urllib.request.urlopen(req, timeout=15) as response:            data = json.loads(response.read().decode("utf-8"))        pages = data.get("query", {}).get("pages", {})        for pid, pdata in pages.items():            if pid == "-1":                return None            return pdata.get("extract", "")    except Exception as e:        print(f"  Error: {e}")        return Nonedef clean_text(text):    if not text:        return ""    text = re.sub(r'\n{3,}', '\n\n', text)    text = re.sub(r'\[\d+\]', '', text)    lines = text.split('\n')    lines = [l for l in lines if len(l.strip()) > 20 or l.strip() == ""]    return '\n'.join(lines).strip()def scrape_wikipedia(topics, max_articles=40):    import time    all_text = []    scraped = set()    count = 0    print(f"Scraping up to {max_articles} Wikipedia articles...")    for topic in topics:        if count >= max_articles:            break        if topic in scraped:            continue        print(f"  [{count+1}/{max_articles}] {topic}", end="")        text = fetch_wikipedia_article(topic)        time.sleep(0.5)        if text:            cleaned = clean_text(text)            if len(cleaned) > 200:                all_text.append(cleaned)                scraped.add(topic)                count += 1                print(f" - {len(cleaned):,} chars")            else:                print(" - too short, skipped")        else:            print(" - not found")    corpus = "\n\n".join(all_text)    print(f"\nTotal corpus: {len(corpus):,} characters")    return corpusprint("Scraper ready!")

## Part 4: Scrape Data & Prepare for Training

In [ ]:
# Scrape Wikipediacorpus = scrape_wikipedia(TECHNOLOGY_TOPICS, max_articles=30)# Tokenize and check sizetokenizer = tiktoken.get_encoding("gpt2")total_tokens = len(tokenizer.encode(corpus))print(f"\nTotal tokens: {total_tokens:,}")print(f"Sample (first 500 chars):\n{corpus[:500]}...")

## Part 5: Train GPT From Scratch on Technology DataUsing a moderate config that trains well on Colab's T4 GPU.

In [ ]:
# Model Config (GPT-2 124M)GPT_Config = {    "vocab_size": 50257,    "context_length": 1024,    "emb_dim": 768,    "n_heads": 12,    "n_layers": 12,    "drop_rate": 0.1,    "qkv_bias": False,}# Split datatrain_ratio = 0.85split_idx = int(train_ratio * len(corpus))train_data = corpus[:split_idx]val_data = corpus[split_idx:]# Create dataloaderstrain_loader = create_dataloader(train_data, tokenizer, batch_size=8,                                  max_len=GPT_Config["context_length"],                                  stride=GPT_Config["context_length"], shuffle=True)val_loader = create_dataloader(val_data, tokenizer, batch_size=8,                                max_len=GPT_Config["context_length"],                                stride=GPT_Config["context_length"], shuffle=False, drop_last=False)print(f"Training batches: {len(train_loader)}")print(f"Validation batches: {len(val_loader)}")# Initialize modeltorch.manual_seed(123)model = GPTModel(GPT_Config).to(device)total_params = sum(p.numel() for p in model.parameters())print(f"Model parameters: {total_params:,}")

In [ ]:
# Train!optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)start_time = time.time()train_losses, val_losses = train_model(    model=model,    train_loader=train_loader,    val_loader=val_loader,    optimizer=optimizer,    device=device,    num_epoch=20,    eval_freq=10,    eval_iter=5,    start_context="Artificial intelligence is",    tokenizer=tokenizer,)elapsed = (time.time() - start_time) / 60print(f"\nTraining completed in {elapsed:.1f} minutes")

## Part 6: Plot Training Progress

In [ ]:
import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(10, 5))ax.plot(train_losses, label="Train Loss")ax.plot(val_losses, label="Val Loss")ax.set_xlabel("Evaluation Step")ax.set_ylabel("Loss")ax.set_title("Training Progress")ax.legend()plt.tight_layout()plt.show()print(f"Final train loss: {train_losses[-1]:.3f}")print(f"Final val loss: {val_losses[-1]:.3f}")

## Part 7: Test the Trained ModelLet's test with various technology-related prompts!

In [ ]:
def generate_response(prompt, model, tokenizer, device, max_tokens=100, temperature=0.7):    model.eval()    input_ids = text_to_token(prompt, tokenizer).to(device)    with torch.no_grad():        output_ids = generate_text(model, input_ids, max_new_token=max_tokens,                                    context_size=model.pos_emb.weight.shape[0],                                    temperature=temperature, top_k=40)    full_text = token_to_text(output_ids, tokenizer)    # Return only the generated part    return full_text[len(prompt):]# Test promptstest_prompts = [    "Artificial intelligence is",    "Machine learning algorithms",    "The future of computing",    "Neural networks can be used",    "Cloud computing enables",    "Programming languages are",    "Data science involves",    "Quantum computing will",]print("=" * 70)print("TESTING TRAINED MODEL")print("=" * 70)for prompt in test_prompts:    response = generate_response(prompt, model, tokenizer, device, max_tokens=80)    print(f"\nPrompt: \"{prompt}\"")    print(f"Output: {prompt}{response}")    print("-" * 70)

## Part 8: Interactive ChatType your own prompts to test the model!

In [ ]:
# Interactive testingprint("=" * 70)print("INTERACTIVE MODE - Type your prompts (type 'quit' to exit)")print("=" * 70)while True:    prompt = input("\nYou: ").strip()    if prompt.lower() in ['quit', 'exit', 'q']:        print("Goodbye!")        break    if not prompt:        continue    response = generate_response(prompt, model, tokenizer, device, max_tokens=100)    print(f"Model: {prompt}{response}")

## Part 9: Save Trained Model

In [ ]:
# Save the trained modelos.makedirs("checkpoints", exist_ok=True)torch.save({    "model_state_dict": model.state_dict(),    "config": GPT_Config,    "train_losses": train_losses,    "val_losses": val_losses,}, "checkpoints/gpt_technology_colab.pt")print("Model saved to checkpoints/gpt_technology_colab.pt")# Download the model (uncomment to download from Colab)# from google.colab import files# files.download("checkpoints/gpt_technology_colab.pt")

---> **Note:** This model was trained from scratch on a small Wikipedia corpus.> For significantly better results, you could fine-tune the pre-trained GPT-2 124M model> (which has been trained on 40GB of internet text). The architecture above is the same> one used in GPT-2 — the difference is just the scale of training data and compute.